In [3]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
import math

########################################
#      PDF Link Collection (CSV)       #
########################################

# Define dataset
df = pd.read_csv('in-the-press-files.csv')
df_copy = df.copy()

# Define the folder path
file_name = 'docs'
file_name_zip = file_name+'.zip'
json_name = 'json'
json_name_zip = file_name+'.zip'

whitelistCharacters = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '-', '.']

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(file_name) or os.path.exists(file_name_zip) is True:
        shutil.rmtree(file_name)
        os.remove(file_name_zip)
    while os.path.exists(json_name) or os.path.exists(json_name_zip) is True:
        shutil.rmtree(json_name)
        os.remove(json_name_zip)
except:
  pass

# Split the folder structure to get the file name via the last element
def processCSV():
    for index, value in enumerate(df_copy['File name']):
        temp = ""
        for name in value:
            if name.lower() not in whitelistCharacters:
                temp+="-"
            else:
                temp+=name.lower()
        df_copy.loc[index, 'renamed'] = temp
    downloadFiles()
    generateSchema()

# Generate schema
def generateSchema():
    for index, value in enumerate(df_copy['File name']):
        data = {
            "version": "0.1.0",
            "layout": "link",
            "page": {
                "title": df_copy["File name"][index],
                # File path can be customised for convenience
                "ref": f"/files/{df_copy['renamed'][index]}{'.docx' if '.docx' in df_copy['File link'][index] else '.doc' if '.doc' in df_copy['File link'][index] else '.xlxs' if '.xlxs' in df_copy['File link'][index] else '.xls' if '.xls' in df_copy['File link'][index] else '.zip' if '.zip' in df_copy['File link'][index] else '.pdf'}",
                "category": df_copy["Category"][index],
                "date": df_copy["Published date"][index]
            },
            "content": []
        }

        # Create the directory if it doesn't exist
        os.makedirs(json_name, exist_ok=True)

        # Create the file path using os.path.join
        file_path = os.path.join(json_name, f"{df_copy['renamed'][index]}.json")

        # Create json file
        with open(f"{file_path}", 'w+', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
            print("Generating", f"{df_copy['renamed'][index]}.json")

# Download files
def downloadFiles():
    for index, value in enumerate(df_copy['File name']):
        # Check if file link is invalid
        page = requests.get(df_copy['File link'][index])
        if page.status_code != 200:
            print(df_copy['File name'][index])
            
        # Create the directory if it doesn't exist
        os.makedirs(file_name, exist_ok=True)
        
        # Create the file path using os.path.join
        file_path = os.path.join(file_name, f"{df_copy['renamed'][index]}{'.docx' if '.docx' in df_copy['File link'][index] else '.doc' if '.doc' in df_copy['File link'][index] else '.xlxs' if '.xlxs' in df_copy['File link'][index] else '.xls' if '.xls' in df_copy['File link'][index] else '.zip' if '.zip' in df_copy['File link'][index] else '.pdf'}")
        
        # Download file from URL
        urllib.request.urlretrieve(df_copy['File link'][index], file_path)

        # Debating between dividing 1024 or 1000. Generally 1 Kilo = 1000(grams), But in Binary 1 Kilo = 1024 bytes
        fileSize = str(math.ceil(os.path.getsize(f"docs/{df_copy['renamed'][index]}{'.docx' if '.docx' in df_copy['File link'][index] else '.doc' if '.doc' in df_copy['File link'][index] else '.xlxs' if '.xlxs' in df_copy['File link'][index] else '.xls' if '.xls' in df_copy['File link'][index] else '.zip' if '.zip' in df_copy['File link'][index] else '.pdf'}")/1024))
        # Calculate and define file size
        fileSize = fileSize + ' KB' if len(fileSize) <= 3 else str(round((int(fileSize)/1000), 2)) + ' MB' if len(fileSize) >= 4 and len(fileSize) < 7 else str(fileSize) + ' B'

        df_copy.loc[index, "File name"] = df_copy["File name"][index] + f" [{'.docx' if '.docx' in df_copy['File link'][index] else 'DOC' if '.doc' in df_copy['File link'][index] else 'XLSX' if '.xlsx' in df_copy['File link'][index] else 'XLS' if '.xls' in df_copy['File link'][index] else 'ZIP' if '.zip' in df_copy['File link'][index] else 'PDF'}, " + fileSize + "]"

processCSV()

Generating joint-press-release--centre-for-liveable-cities-and-ramboll-launch-an-urban-lab-to-put-regenerative-principles-at-the-heart-of-city-planning-.json
Generating media-factsheet--dementia-friendly-neighbourhood-study.json
Generating media-release-on-temasek-foundation-leaders-for-urban-governance-programme.json
Generating no-better-time-to-build-urban-resilience.-experts-work-with-asian-cities-to-advance-actionable-project-proposals-in-inaugural-programme-aimed-at-strengthening-resilience-.json
Generating joint-press-release--singapore-and-un-habitat-aim-to-support-up-to-a-hundred-cities-in-achieving-their-urban-development-and-sustainability-goals-by-2030.json
Generating press-release--new-webinar-series-by-the-centre-for-liveable-cities--live-conversations-with-global-urban-leaders-on-how-to-navigate-and-respond-to-challenges-posed-by-disruptions.json


In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests

########################################
#        Links Collection (CSV)        #
########################################

# Define dataset
df = pd.read_csv('csa-security-bulletin.csv')
df_copy = df.copy()

# Define statuses to be skipped if any
exempt = ['Migrated', 'Skipped']

# Define the folder path
file_name = 'docs'
file_name_zip = file_name+'.zip'

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(file_name) or os.path.exists(file_name_zip) is True:
        shutil.rmtree(file_name)
        os.remove(file_name_zip)
except:
  pass

def generateSchema():
    # Clean title to use as JSON file name
    for index, value in enumerate(df_copy['Title']):
        replacedString = value.replace(".", "")
        replacedString = replacedString.replace(" ", "-")
        replacedString = replacedString.replace(",", "-")
        replacedString = replacedString.replace("(", "-")
        replacedString = replacedString.replace(":", "-")
        replacedString = replacedString.replace("%", "-")
        replacedString = replacedString.replace("?", "-")
        replacedString = replacedString.replace('"', "-")
        replacedString = replacedString.replace("、", "-")
        replacedString = replacedString.replace("&", "-")
        replacedString = replacedString.replace(")", "-")
        replacedString = replacedString.replace("|", "-")
        replacedString = replacedString.replace("/", "-")
        replacedString = replacedString.replace("\\", "-")
        replacedString = replacedString.lower()
        df_copy.loc[index, 'renamed'] = replacedString

        # Generate schema
        try:
            data = {
              "version": "0.1.0",
              "layout": "link",
              "page": {
                "title": f"{df_copy['Title'][index]}",
                # Use if fileName is like /folder1/folder2/folder3/document.pdf
                # "ref": f"{df_copy['fileName'][index].split("/")[-1]}",
                "ref": f"/files/bulletins/{df_copy['fileName'][index]}",
                "category": "Bulletins",
                "date": f"{df_copy["Date"][index]}"
              },
              "content": []
            }
        except:
            print("Error:", df_copy['Title'][index])

        # Create the directory if it doesn't exist
        os.makedirs(file_name, exist_ok=True)

        # Create the file path using os.path.join
        file_path = os.path.join(file_name, f"{df_copy['renamed'][index]}.json")
        
        # Add index to name to prevent duplication
        if os.path.exists(file_path):
            file_path = os.path.join(file_name, f"{df_copy['renamed'][index]}-{index}.json")

        # Create json file
        with open(f"{file_path}", 'w+', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)

generateSchema()
print("Done")

In [ ]:
import requests
from requests.auth import HTTPBasicAuth
from urllib.parse import urljoin, urlparse
from dotenv import load_dotenv
import os
import json
from bs4 import BeautifulSoup
from requests.exceptions import SSLError, ConnectionError, RequestException, Timeout
from urllib3.exceptions import NewConnectionError, NameResolutionError
import pandas as pd
import ssl

########################################
#          Broken link checker         #
########################################

# Load the .env file that stores secrets
load_dotenv()

# Extract password
password = os.getenv("IPOS")

# Define staging site
domain = "https://staging.d3t3m6no0k8kp4.amplifyapp.com"
url = domain + "/sitemap.json"

# Get staging site's sitemap
page = requests.get(url, auth=HTTPBasicAuth('user', password))
pageJson = json.loads(page.text)

# List to store path of sites
path = []

# Exclude all collection as children pages are already on the sitemap
collection = []

# Traverse the sitemap and collect paths and collections, considering nested structures
for index, value in enumerate(pageJson['children']):
    try:
        if pageJson['children'][index]["layout"] == "collection":
            collection.append(pageJson['children'][index]["permalink"].split("/")[-1])
        if 'ref' in pageJson['children'][index]:
            path.append(pageJson['children'][index]['ref'])
        else:
            path.append(pageJson['children'][index]['permalink'])
            if 'children' in pageJson['children'][index]:
                for i in range(0, len(pageJson['children'][index]['children'])):
                    if pageJson['children'][index]["children"][i]["layout"] == "collection":
                        collection.append(pageJson['children'][index]["children"][i]["permalink"].split("/")[-1])
                    if 'ref' in pageJson['children'][index]['children'][i]:
                        path.append(pageJson['children'][index]['children'][i]['ref'])
                    else:
                        path.append(pageJson['children'][index]['children'][i]['permalink'])
                        if 'children' in pageJson['children'][index]['children'][i]:
                            for c in range(0, len(pageJson['children'][index]['children'][i]['children'])):
                                if pageJson['children'][index]['children'][i]["children"][c]["layout"] == "collection":
                                    temp = pageJson['children'][index]['children'][i]["children"][c]["permalink"].split("/")[-1]
                                    if temp not in collection:
                                        collection.append(temp)
                                if 'ref' in pageJson['children'][index]['children'][i]['children'][c]:
                                    path.append(pageJson['children'][index]['children'][i]['children'][c]['ref'])
                                else:
                                    path.append(pageJson['children'][index]['children'][i]['children'][c]['permalink'])
                                    if 'children' in pageJson['children'][index]['children'][i]['children'][c]:
                                        for y in range(0, len(pageJson['children'][index]['children'][i]['children'][c]['children'])):
                                            if pageJson['children'][index]['children'][i]["children"][c]["children"][y]["layout"] == "collection":
                                                temp = pageJson['children'][index]['children'][i]['children'][c]["children"][y]['permalink'].split("/")[-1]
                                                if temp not in collection:
                                                    collection.append(temp)                                                
                                            if 'ref' in pageJson['children'][index]['children'][i]['children'][c]['children'][y]:
                                                path.append(pageJson['children'][index]['children'][i]['children'][c]['children'][y]['ref'])
                                            else:
                                                path.append(pageJson['children'][index]['children'][i]['children'][c]['children'][y]['permalink'])

    except Exception as e:
        pass

print(len(path), "pages")
print(collection)

# Dictionary to store error pages and counter for error entries
errorPages = {}
count = 0

# Iterate through all paths and check for broken links
for i, v in enumerate(path):
    if v == "/search":
        continue
    internalLinks = []

    # Check external links for errors
    if "https://" in v or "http://" in v or "www." in v:
        try:
            resp = requests.get(i, allow_redirects=True)
            if resp.status_code == 404:
                errorPages[count] = {"Staging": v}
                count += 1
                print("Link:", v, "404!")
        except NewConnectionError:
            errorPages[count] = {"Staging": v}
            count += 1
            print("Link:", v, "404!")
        except requests.exceptions.RequestException as e:
            errorPages[count] = {"Staging": v}
            count += 1
            print("Link:", v, "404!")
        
    # Check internal links for errors
    else:
        url = domain + v
        # Request header data of staging URL
        page = requests.head(url, auth=HTTPBasicAuth('user', password), allow_redirects=True)

        # Check if it redirects to a 404 page
        if page.url != domain + "/404.html":
            # Ignore files and images - they are covered in the workflow below
            if "/files" in v or "/images" in v:
                continue
            elif "/files" not in v or "/images" not in v:
                # Proceed if the filepath is not a collection
                if v.split("/")[-1] not in collection:
                    # Fetch HTML of staging URL
                    page = requests.get(url, auth=HTTPBasicAuth('user', password), allow_redirects=True)
                    soup = BeautifulSoup(page.content, "html.parser")

                    # Try and find all links within available div classes (different components) and append to links list
                    # New classes will be added iteratively
                    try:
                        content = soup.find("div", class_ = "col-span-12 flex flex-col gap-16 lg:col-span-9 lg:mr-24")
                        links = content.find_all("a")
                    except AttributeError:
                        try:
                            content = soup.find("div", class_ = "grid grid-cols-1 gap-10 md:gap-7 lg:gap-x-16 lg:gap-y-12")
                            links = content.find_all("a")
                        except AttributeError:
                            try:
                                content = soup.find("div", class_ = "col-span-12 flex flex-col gap-16 max-w-[54rem]")
                                links = content.find_all("a")
                            except AttributeError:
                                try:
                                    content = soup.find("div", class_ = "mx-auto grid max-w-screen-xl grid-cols-12 px-6 py-12 md:px-10 md:py-16 lg:gap-6 xl:gap-10")
                                    links = content.find_all("a")
                                except:
                                    content = soup.find("div", class_ = "w-full overflow-x-auto lg:max-w-[660px]")
                                    links = content.find_all("a")
                                
                    # Loop through all links found in the staging URL
                    for index, value in enumerate(links):
                        try:
                            # Workflow for external link
                            if "https://" in value["href"] or "http://" in value["href"]:
                                try:
                                    resp = requests.get(value["href"], allow_redirects=True)
                                    if resp.status_code == 404:
                                        errorPages[count] = {"Staging": url, "href": value["href"]}
                                        count += 1
                                        print("Staging:", url, "href:", value["href"], "404!")
                                except NewConnectionError:
                                    errorPages[count] = {"Staging": url, "href": value["href"]}
                                    count += 1
                                    print("Staging:", url, "href:", value["href"], "404!")
                                except requests.exceptions.RequestException as e:
                                    errorPages[count] = {"Staging": url, "href": value["href"]}
                                    count += 1
                                    print("Staging:", url, "href:", value["href"], "404!")
                                
                            # Check if it contains a empty href
                            if "undefined" in value["href"]:
                                errorPages[count] = {"Staging": domain+v}
                                count += 1
                                print("Staging:", domain+v, "Empty href!", "404!")

                            # Exclude all mail and telephone href
                            if value["href"][0] != "#" and "mailto:" not in value["href"] and "tel:" not in value["href"]:
                                page = requests.head(value["href"] if "https" in value["href"] or "http" in value["href"] else domain + value["href"], auth=HTTPBasicAuth('user', password), allow_redirects=True)
                                # Checks if it leads to error pages - Good to know, but it shouldnt be our problem
                                # You can choose to ignore this output or inform your agency
                                if "PageNotFound" in page.url or page.url == domain + "/404.html" or "closepage" in page.url or page.url == domain + "/undefined":
                                    errorPages[count] = {"Staging": domain+v, "href": value["href"], "Redirected": page.url}
                                    count += 1
                                    print("Staging:", domain+v, "href:", value["href"], "Redirected:", page.url, "404!")
                                if page.url != domain + "/404.html":
                                    continue
                                if page.url == domain + "/404.html":
                                    errorPages[count] = {"Staging": url, "href": value["href"]}
                                    count += 1
                                    print("Staging:", url, "href:", value["href"], "Page 404!")
                            else:
                                continue

                        # Catch generic site errors
                        except SSLError:
                            errorPages[count] = {"Staging": domain+v, "href": value["href"], "Redirected": page.url}
                            count += 1
                            print("Staging:", domain+v, "href:", value["href"], "Redirected:", page.url, "404, SSL Error")
                        except requests.exceptions.Timeout:
                            errorPages[count] = {"Staging": domain+v, "href": value["href"], "Redirected": page.url}
                            count += 1
                            print("Staging:",domain+v, "href:", value["href"], "Redirected:", page.url, "404, Timeout")
                        except ssl.SSLCertVerificationError:
                            errorPages[count] = {"Staging": domain+v, "href": value["href"], "Redirected": page.url}
                            count += 1
                            print("Staging:",domain+v, "href:", value["href"], "Redirected:", page.url, "404!")
                        except requests.exceptions.HTTPError:
                            errorPages[count] = {"Staging": domain+v, "href": value["href"], "Redirected": page.url}
                            count += 1
                            print("Staging:",domain+v, "href:", value["href"], "Redirected:", page.url, "404, HTTP Error")
                        except requests.exceptions.ConnectionError:
                            errorPages[count] = {"Staging": domain+v, "href": value["href"], "Redirected": page.url}
                            count += 1
                            print("Staging:",domain+v, "href:", value["href"], "Redirected:", page.url, "404, Connection Error")
                        except NameResolutionError:
                            errorPages[count] = {"Staging": domain+v, "href": value["href"], "Redirected": page.url}
                            count += 1
                            print("Staging:",domain+v, "href:", value["href"], "Redirected:", page.url, "404, Resolution Error")
                        except Exception as e:
                            errorPages[count] = {"Staging": domain+v, "href": value["href"], "Redirected": page.url}
                            count += 1
                            print(url, value["href"], e)

        else:
            errorPages[count] = {"Staging": url, "href": value["href"]}
            count += 1
            print("Staging:", url, "href:", value["href"], "Page 404!")

# Generate report in .csv
(pd.DataFrame.from_dict(data=errorPages, orient='index')
   .to_csv('ErrorPages.csv', header=True))